##### Import des bibliothèques

In [2]:
import base64
import json
import time

from IPython.display import Markdown
from openai import OpenAI
from pathlib import Path

##### Premier test avec une image et Ollama

In [3]:
MODEL = 'qwen3-vl:4b-instruct'
BASE_URL = 'http://localhost:11434/v1'
API_KEY = 'ollama'
IMG_DIR = '../atelier_scanner_etiquettes/data/photos'
TEMPERATURE = 0

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)

photo = Path(IMG_DIR) / '10_3608580065340_etiquette.jpg'
encoded_image = base64.b64encode(photo.read_bytes()).decode('ascii')

url_image = f'data:image/jpeg;base64,{encoded_image}'

prompt = """Retourne moi les informations du produit et ses informations
nutritionnelles au format JSON.

Si tu n'as pas les informations pour identifier le produit, n'invente rien.
Si tu ne trouves pas de valeurs pour un nutriment, mets null.

Voici le format attendu :

{
  "statut": "verifie" | "non_verifie" | "illisible",
  "produit": {
    "nom": <texte> | null, 
    "marque": <texte> | null, 
    "code_barres": <texte> | null
    },
  "base": "100 g" | "100 ml" | null,
  "nutriments": {
    "kj", 
    "kcal", 
    "lipides", 
    "satures", 
    "glucides", 
    "sucres", 
    "fibres", 
    "proteines", 
    "sel"
    } : nombre ou null,
  "ingredients": [<texte>, ...] | null,
}
"""

messages = [{'role': 'user', 'content': [
    {'type': 'text', 'text': prompt},
    {'type': 'image_url', 'image_url': {'url': url_image}}
]}]

debut = time.perf_counter()
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=TEMPERATURE
)

print(response.choices[0].message.content)

latence = time.perf_counter() - debut

print(f'{latence:.1f} s,',
        response.usage.prompt_tokens, 'tokens en entrée,',
        response.usage.completion_tokens, 'en sortie')

{
  "statut": "non_verifie",
  "produit": {
    "nom": null,
    "marque": null,
    "code_barres": null
  },
  "base": "100 g",
  "nutriments": {
    "kj": 2299,
    "kcal": 551,
    "lipides": 34,
    "satures": 5.2,
    "glucides": 53,
    "sucres": 51,
    "fibres": 3.3,
    "proteines": 6.7,
    "sel": 0.1
  },
  "ingredients": null
}
5.2 s, 1279 tokens en entrée, 156 en sortie


##### Affichage des mesures

In [4]:
Markdown(f'''
##### Essai

- Modèle : `{MODEL}`
- Latence : {latence:.1f} s
- Tokens en entrée : {response.usage.prompt_tokens}
- Tokens en sortie : {response.usage.completion_tokens}
'''
)


##### Essai

- Modèle : `qwen3-vl:4b-instruct`
- Latence : 5.2 s
- Tokens en entrée : 1279
- Tokens en sortie : 156


##### Test sur une photo de face

In [5]:
photo = Path(IMG_DIR) / '10_3608580065340_face.jpg'
encoded_image = base64.b64encode(photo.read_bytes()).decode('ascii')

url_image = f'data:image/jpeg;base64,{encoded_image}'

messages = [{'role': 'user', 'content': [
    {'type': 'text', 'text': 'Donne moi la marque et le modèle de ce produit'},
    {'type': 'image_url', 'image_url': {'url': url_image}}
]}]

debut = time.perf_counter()
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=TEMPERATURE
)

print(response.choices[0].message.content)

latence = time.perf_counter() - debut

print(f'{latence:.1f} s,',
        response.usage.prompt_tokens, 'tokens en entrée,',
        response.usage.completion_tokens, 'en sortie')

La marque de ce produit est **Bonne Maman**.

Le modèle ou le nom du produit est : **pâte à tartiner noisettes et Cacao**.

C’est une pâte à tartiner au goût de noisettes et de cacao, dans un emballage en verre typique de la marque, avec un couvercle à motif carreaux rouges et blancs.
1.9 s, 1079 tokens en entrée, 86 en sortie


In [11]:
%load_ext autoreload
%autoreload 2

from nutriscope.scanner import scanner_etiquette

resultat = scanner_etiquette(Path('../atelier_scanner_etiquettes/data/photos/19_87157215_etiquette.jpg'))

resultat


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


{'statut': 'non_verifie',
 'produit': {'nom': None, 'marque': None, 'code_barres': None},
 'base': '100 g',
 'nutriments': {'kj': 435,
  'kcal': 102,
  'lipides': 0.1,
  'satures': 0.1,
  'glucides': 23.2,
  'sucres': 22.8,
  'fibres': None,
  'proteines': 1.2,
  'sel': 1.8},
 'inferieur_a': ['satures'],
 'ingredients': None,
 'fiche_catalogue': None,
 'ecarts': [],
 'alertes': [],
 'mesures': {'appels_modele': 1,
  'latence_s': 5.3,
  'tokens_entree': 1484,
  'tokens_sortie': 176}}

In [17]:
from pathlib import Path
from nutriscope.scanner import scanner_etiquette

photo = Path('../atelier_scanner_etiquettes/data/photos/19_87157215_etiquette.jpg')
result = scanner_etiquette(photo)

print("Nutriments retournés : ")
for cle, valeur in result['nutriments'].items():
    print(f'{cle}: {valeur}')

print('\nAlertes :')
print(result['alertes'])

print('\nStatut :')
print(result['statut'])

Nutriments retournés : 
kj: 435
kcal: 102
lipides: 0.1
satures: 0.1
glucides: 23.2
sucres: 22.8
fibres: None
proteines: 1.2
sel: 1.8

Alertes :
[]

Statut :
non_verifie


In [18]:
import json

mauvais_json = {
    "statut": "non_verifie",
    "produit": {"nom": "Test", "marque": None, "code_barres": None},
    "base": "100 g",
    "nutriments": {
        "kj": 435,
        "kcal": 102,
        "lipides": 0,
        "satures": 0.1,
        "glucides": 23.2,
        "sucres": 0,
        "fibres": None,
        "proteines": 1.2,
        "sel": 1.8
    },
    "inferieur_a": ["satures"],
    "ingredients": None,
    "alertes": []
}

print("Les zéros à glucides et sucres sont INVENTÉS")
print(f"inferieur_a: {mauvais_json['inferieur_a']}")

Les zéros à glucides et sucres sont INVENTÉS
inferieur_a: ['satures']
